## Import libraries and load source data

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Feature Engineering for Fantasy Football ML Model")
print("="*80)
print("\nThis notebook transforms raw stats into ML-ready features.")
print("Key principle: All historical metrics are SHIFTED by 1 week to prevent data leakage.\n")

## Install nfl_data_py library

In [0]:
# Install required packages for NFL data extraction
%pip install --no-deps nfl_data_py
%pip install appdirs fastparquet pandas

## Run nfl_py_extraction notebook to get source data

In [0]:
# Load source data directly using nfl_data_py library
import nfl_data_py as nfl
import pandas as pd

print("Loading data from nfl_data_py library...")

SEASONS = [2020, 2021, 2022, 2023, 2024, 2025]

# Load play-by-play data for all seasons
pbp = nfl.import_pbp_data(SEASONS, downcast=True)
print(f"Loaded {len(pbp)} plays from seasons {SEASONS}")

# Load schedules
schedules = nfl.import_schedules(SEASONS)
print(f"Loaded {len(schedules)} games from schedules")

# Load snap counts (for snap share)
snap_counts = nfl.import_snap_counts(SEASONS)
print(f"Loaded {len(snap_counts)} snap count records")

# Load injury reports (for supporting cast health)
injuries = nfl.import_injuries(SEASONS)
print(f"Loaded {len(injuries)} injury records")

# Load rosters (official positions incl. TE, and PFR->GSIS id mapping)
rosters = nfl.import_seasonal_rosters(SEASONS)
print(f"Loaded {len(rosters)} roster records")

# Load depth charts. The schema differs by era:
#   2023-2024: weekly lists with a depth_team rank column
#   2025+:     daily snapshots with dt/pos_rank columns
depth_charts_weekly = nfl.import_depth_charts([s for s in SEASONS if s <= 2024])
depth_charts_snapshots = nfl.import_depth_charts([s for s in SEASONS if s >= 2025])
print(f"Loaded {len(depth_charts_weekly)} weekly + {len(depth_charts_snapshots)} snapshot depth chart records")

# Calculate weekly stats (from the extraction notebook logic)
print("\nCalculating weekly player statistics from play-by-play data...")

# Passer aggregations
passing_plays = pbp[pbp['play_type'] == 'pass'].copy()
passer_stats = passing_plays.groupby(
    ['season', 'week', 'passer_player_id', 'passer_player_name'],
    dropna=False
).agg({
    'pass_attempt': 'sum',
    'complete_pass': 'sum',
    'yards_gained': 'sum',
    'pass_touchdown': 'sum',
    'interception': 'sum'
}).reset_index()
passer_stats.rename(columns={
    'passer_player_id': 'player_id',
    'passer_player_name': 'player_name',
    'pass_attempt': 'pass_attempts',
    'complete_pass': 'completions',
    'yards_gained': 'passing_yards',
    'pass_touchdown': 'passing_tds',
    'interception': 'interceptions'
}, inplace=True)

# Rusher aggregations
rushing_plays = pbp[pbp['play_type'] == 'run'].copy()
rusher_stats = rushing_plays.groupby(
    ['season', 'week', 'rusher_player_id', 'rusher_player_name'],
    dropna=False
).agg({
    'rush_attempt': 'sum',
    'yards_gained': 'sum',
    'rush_touchdown': 'sum'
}).reset_index()
rusher_stats.rename(columns={
    'rusher_player_id': 'player_id',
    'rusher_player_name': 'player_name',
    'rush_attempt': 'rush_attempts',
    'yards_gained': 'rushing_yards',
    'rush_touchdown': 'rushing_tds'
}, inplace=True)

# Receiver aggregations
receiving_plays = pbp[
    (pbp['play_type'] == 'pass') & 
    (pbp['receiver_player_id'].notna())
].copy()
receiver_stats = receiving_plays.groupby(
    ['season', 'week', 'receiver_player_id', 'receiver_player_name'],
    dropna=False
).agg({
    'pass_attempt': 'sum',
    'complete_pass': 'sum',
    'yards_gained': 'sum',
    'pass_touchdown': 'sum'
}).reset_index()
receiver_stats.rename(columns={
    'receiver_player_id': 'player_id',
    'receiver_player_name': 'player_name',
    'pass_attempt': 'targets',
    'complete_pass': 'receptions',
    'yards_gained': 'receiving_yards',
    'pass_touchdown': 'receiving_tds'
}, inplace=True)

# Merge all stats
weekly_stats = passer_stats.merge(rusher_stats, on=['season', 'week', 'player_id', 'player_name'], how='outer')
weekly_stats = weekly_stats.merge(receiver_stats, on=['season', 'week', 'player_id', 'player_name'], how='outer')
weekly_stats['player_name'] = weekly_stats['player_name'].ffill().bfill()

numeric_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds'
]
weekly_stats[numeric_cols] = weekly_stats[numeric_cols].fillna(0)

# Calculate fantasy points (PPR)
weekly_stats['fantasy_points_ppr'] = (
    (weekly_stats['passing_yards'] * 0.04) +
    (weekly_stats['passing_tds'] * 4) +
    (weekly_stats['interceptions'] * -2) +
    (weekly_stats['rushing_yards'] * 0.1) +
    (weekly_stats['rushing_tds'] * 6) +
    (weekly_stats['receiving_yards'] * 0.1) +
    (weekly_stats['receiving_tds'] * 6) +
    (weekly_stats['receptions'] * 1)
)
weekly_stats['fantasy_points_ppr'] = weekly_stats['fantasy_points_ppr'].round(2)
weekly_stats = weekly_stats.sort_values('fantasy_points_ppr', ascending=False).reset_index(drop=True)

print(f"\n✓ Data loaded and ready for feature engineering:")
print(f"  - weekly_stats: {len(weekly_stats):,} player-week records")
print(f"  - schedules: {len(schedules):,} games")
print(f"  - pbp: {len(pbp):,} plays across {weekly_stats['season'].nunique()} seasons")

## Identify player positions and teams from PBP data

In [0]:
# ============================================================================
# STEP 1: Identify player positions and teams from play-by-play data
# ============================================================================
print("Step 1: Identifying player positions and teams...\n")

# Extract position from PBP data (passers = QB, rushers = RB, receivers = WR/TE)
passers = pbp[pbp['passer_player_id'].notna()][['passer_player_id', 'passer_player_name', 'posteam', 'season', 'week']].copy()
passers.columns = ['player_id', 'player_name', 'team', 'season', 'week']
passers['position'] = 'QB'

rushers = pbp[pbp['rusher_player_id'].notna()][['rusher_player_id', 'rusher_player_name', 'posteam', 'season', 'week']].copy()
rushers.columns = ['player_id', 'player_name', 'team', 'season', 'week']
rushers['position'] = 'RB'  # Default to RB, we'll refine this

receivers = pbp[pbp['receiver_player_id'].notna()][['receiver_player_id', 'receiver_player_name', 'posteam', 'season', 'week']].copy()
receivers.columns = ['player_id', 'player_name', 'team', 'season', 'week']
receivers['position'] = 'WR'  # Default to WR, we'll refine based on usage patterns

# Combine all players
all_players = pd.concat([passers, rushers, receivers], ignore_index=True)

# For each player, take the most common position (mode)
player_position = all_players.groupby('player_id').agg({
    'player_name': 'first',
    'position': lambda x: x.mode()[0] if not x.mode().empty else 'FLEX'
}).reset_index()

# Override the heuristic with official roster positions where available.
# This adds TE (the heuristic can't detect it) and fixes pass-catching RBs
# that would otherwise be tagged WR. Keep each player's latest-season position.
roster_positions = (
    rosters.dropna(subset=['player_id', 'position'])
    .sort_values('season')
    .drop_duplicates('player_id', keep='last')
)[['player_id', 'position']]
roster_positions.columns = ['player_id', 'roster_position']

player_position = player_position.merge(roster_positions, on='player_id', how='left')
use_roster = player_position['roster_position'].isin(['QB', 'RB', 'WR', 'TE', 'FB'])
player_position.loc[use_roster, 'position'] = (
    player_position.loc[use_roster, 'roster_position'].replace({'FB': 'RB'})
)
player_position.drop(columns=['roster_position'], inplace=True)

print(f"Identified positions for {len(player_position)} unique players:")
print(player_position['position'].value_counts())

# Get each player's team PER SEASON (players change teams between years,
# so a single "most recent team" would corrupt older seasons' joins)
player_team = (
    all_players.sort_values(['season', 'week'])
    .groupby(['player_id', 'season'])
    .agg({'team': 'last'})
    .reset_index()
)
player_team.columns = ['player_id', 'season', 'recent_team']

# Merge position and team into weekly_stats
weekly_stats_df = weekly_stats.copy()
weekly_stats_df = weekly_stats_df.merge(player_position, on='player_id', how='left', suffixes=('', '_lookup'))
weekly_stats_df = weekly_stats_df.merge(player_team, on=['player_id', 'season'], how='left')

# Use lookup values to fill missing player_name if needed
weekly_stats_df['player_name'] = weekly_stats_df['player_name'].fillna(weekly_stats_df['player_name_lookup'])
weekly_stats_df.drop(columns=['player_name_lookup'], inplace=True, errors='ignore')

print(f"\nEnriched weekly_stats with position and team data.")
print(f"Shape: {weekly_stats_df.shape}")
display(weekly_stats_df.head())

## CATEGORY 1: General Game Context Features

In [0]:
# ============================================================================
# CATEGORY 1: General Game Context (Applies to all players)
# ============================================================================
print("\nStep 2: Engineering General Game Context Features...\n")

# Prepare schedules data
schedules_df = schedules.copy()

# Calculate implied team totals.
# nflverse convention: spread_line is POSITIVE when the HOME team is favored
# (e.g. spread_line = 3.5 means home team favored by 3.5).
schedules_df['home_implied_total'] = (schedules_df['total_line'] / 2) + (schedules_df['spread_line'] / 2)
schedules_df['away_implied_total'] = (schedules_df['total_line'] / 2) - (schedules_df['spread_line'] / 2)

# Vegas win probability from moneylines (American odds -> implied probability).
# Note: historical PLAYER prop lines are not freely available (nfl_data_py has
# none; archives are paid APIs), so game-level Vegas (spread, total, moneyline)
# is the full betting-market signal we can use.
import numpy as np

def moneyline_to_prob(ml):
    return np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100))

schedules_df['home_win_prob'] = moneyline_to_prob(schedules_df['home_moneyline'])
schedules_df['away_win_prob'] = moneyline_to_prob(schedules_df['away_moneyline'])
# Fill missing moneylines with a coin flip (rare)
schedules_df['home_win_prob'] = schedules_df['home_win_prob'].fillna(0.5)
schedules_df['away_win_prob'] = schedules_df['away_win_prob'].fillna(0.5)

# Weather flags.
# Note: precipitation is NOT available in nfl_data_py schedules (only temp/wind/roof),
# so is_dome captures weather-proof venues instead.
schedules_df['is_dome'] = schedules_df['roof'].isin(['dome', 'closed']).astype(int)
schedules_df['is_bad_weather'] = (
    ((schedules_df['wind'] > 15) | (schedules_df['temp'] < 32)) &
    (schedules_df['is_dome'] == 0)
).astype(int)

# Create home/away context for each team
home_context = schedules_df[['season', 'week', 'home_team', 'home_implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']].copy()
home_context.columns = ['season', 'week', 'team', 'implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']
home_context['is_home'] = 1
home_context['opponent'] = schedules_df['away_team']
home_context['team_spread'] = schedules_df['spread_line']    # positive = this team favored
home_context['team_win_prob'] = schedules_df['home_win_prob']
home_context['starting_qb_id'] = schedules_df['home_qb_id']

away_context = schedules_df[['season', 'week', 'away_team', 'away_implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']].copy()
away_context.columns = ['season', 'week', 'team', 'implied_total', 'temp', 'wind', 'is_bad_weather', 'is_dome', 'gameday']
away_context['is_home'] = 0
away_context['opponent'] = schedules_df['home_team']
away_context['team_spread'] = -schedules_df['spread_line']   # positive = this team favored
away_context['team_win_prob'] = schedules_df['away_win_prob']
away_context['starting_qb_id'] = schedules_df['away_qb_id']

# Combine home and away
game_context = pd.concat([home_context, away_context], ignore_index=True)

# Calculate rest advantage (days since last game, reset each season so the
# offseason gap doesn't count as rest)
game_context['gameday'] = pd.to_datetime(game_context['gameday'])
game_context = game_context.sort_values(['team', 'gameday'])
game_context['days_since_last_game'] = game_context.groupby(['team', 'season'])['gameday'].diff().dt.days

# Merge opponent's days since last game to calculate rest advantage
opponent_rest = game_context[['season', 'week', 'team', 'days_since_last_game']].copy()
opponent_rest.columns = ['season', 'week', 'opponent', 'opp_days_since_last_game']

game_context = game_context.merge(opponent_rest, on=['season', 'week', 'opponent'], how='left')
game_context['rest_advantage'] = game_context['days_since_last_game'] - game_context['opp_days_since_last_game']
game_context['rest_advantage'] = game_context['rest_advantage'].fillna(0)

# Merge game context into weekly_stats_df
weekly_stats_df = weekly_stats_df.merge(
    game_context[['season', 'week', 'team', 'implied_total', 'team_spread', 'team_win_prob', 'is_home', 'temp', 'wind',
                  'is_bad_weather', 'is_dome', 'rest_advantage', 'opponent', 'starting_qb_id', 'gameday']],
    left_on=['season', 'week', 'recent_team'],
    right_on=['season', 'week', 'team'],
    how='left'
)

# Drop duplicate team column
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

print("✓ Added game context features:")
print("  - implied_total (Vegas implied team total)")
print("  - team_spread (Vegas spread from team's perspective, positive = favored)")
print("  - team_win_prob (implied win probability from the moneyline)")
print("  - is_home (1=home, 0=away)")
print("  - temp, wind (weather conditions)")
print("  - is_bad_weather (wind>15mph or temp<32F, outdoor games only)")
print("  - is_dome (weather-proof venue; precipitation not available in nfl_data_py)")
print("  - rest_advantage (days since last game difference)")
print("  - starting_qb_id (scheduled starting QB, used for QB context features)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 2: Rolling Averages (All Players)

In [0]:
# ============================================================================
# CATEGORY 2: Rolling Averages (3-Week & 5-Week) - For ALL Players
# ============================================================================
print("\nStep 3: Calculating Rolling Averages (SHIFTED to prevent data leakage)...\n")

# Sort by player, season and week to ensure proper rolling calculations.
# Rolling windows reset each season - last year's December form says little
# about this year's September.
weekly_stats_df = weekly_stats_df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)

# Calculate SHIFTED rolling averages for fantasy points
weekly_stats_df['fantasy_points_3wk_avg'] = (
    weekly_stats_df.groupby(['player_id', 'season'])['fantasy_points_ppr']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df['fantasy_points_5wk_avg'] = (
    weekly_stats_df.groupby(['player_id', 'season'])['fantasy_points_ppr']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# -----------------------------------------------------------------------------
# Previous-season carryover features.
# Rolling averages are all 0 in week 1 (they reset each season), so these give
# the model a leakage-free prior for early-season and week-1 predictions.
# -----------------------------------------------------------------------------
season_totals = (
    weekly_stats_df.groupby(['player_id', 'season'])
    .agg(prev_season_ppg=('fantasy_points_ppr', 'mean'),
         prev_season_games=('fantasy_points_ppr', 'size'))
    .reset_index()
)
season_totals['season'] = season_totals['season'] + 1  # shift forward: 2024 stats describe 2025 rows

weekly_stats_df = weekly_stats_df.merge(season_totals, on=['player_id', 'season'], how='left')
# Rookies / first loaded season have no prior year -> 0
weekly_stats_df['prev_season_ppg'] = weekly_stats_df['prev_season_ppg'].fillna(0)
weekly_stats_df['prev_season_games'] = weekly_stats_df['prev_season_games'].fillna(0)

print("✓ Added rolling averages for ALL players:")
print("  - fantasy_points_3wk_avg (shifted 3-week average)")
print("  - fantasy_points_5wk_avg (shifted 5-week average)")
print("  - prev_season_ppg / prev_season_games (previous season carryover, for cold starts)")
print("\n  Note: All rolling metrics use .shift(1) to prevent data leakage.")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 3: QB-Specific Features

In [0]:
# ============================================================================
# CATEGORY 3: Quarterback (QB) Specific Features
# ============================================================================
print("\nStep 4: Engineering QB-Specific Features...\n")

# Filter for QBs only
qb_mask = weekly_stats_df['position'] == 'QB'

# Calculate SHIFTED rolling averages for passing attempts and rushing yards
weekly_stats_df.loc[qb_mask, 'qb_pass_attempts_3wk_avg'] = (
    weekly_stats_df[qb_mask].groupby(['player_id', 'season'])['pass_attempts']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_pass_attempts_5wk_avg'] = (
    weekly_stats_df[qb_mask].groupby(['player_id', 'season'])['pass_attempts']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_rushing_yards_3wk_avg'] = (
    weekly_stats_df[qb_mask].groupby(['player_id', 'season'])['rushing_yards']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[qb_mask, 'qb_rushing_yards_5wk_avg'] = (
    weekly_stats_df[qb_mask].groupby(['player_id', 'season'])['rushing_yards']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- Sack rates (pressure context) ---
# Opponent sack rate = how often the opposing defense sacks the QB (matchup).
# Team sack rate allowed = how often this player's own O-line gives up sacks
# (proxy for O-line strength, since O-line rankings are premium data).
# Both are SHIFTED season-to-date rates: cumulative sacks / cumulative dropbacks
# through the previous week.
dropbacks = pbp[pbp['qb_dropback'] == 1]

def shifted_sack_rate(df, team_col, rate_name):
    g = df.groupby([team_col, 'season', 'week']).agg(
        dropbacks=('qb_dropback', 'sum'),
        sacks=('sack', 'sum')
    ).reset_index().sort_values(['season', 'week'])
    cum_db = g.groupby([team_col, 'season'])['dropbacks'].transform(lambda x: x.shift(1).expanding().sum())
    cum_sk = g.groupby([team_col, 'season'])['sacks'].transform(lambda x: x.shift(1).expanding().sum())
    g[rate_name] = (cum_sk / cum_db).fillna(0)
    return g[[team_col, 'season', 'week', rate_name]]

off_sack_rate = shifted_sack_rate(dropbacks, 'posteam', 'team_sack_rate_allowed')
def_sack_rate = shifted_sack_rate(dropbacks, 'defteam', 'opp_def_sack_rate')

weekly_stats_df = weekly_stats_df.merge(
    off_sack_rate, left_on=['recent_team', 'season', 'week'], right_on=['posteam', 'season', 'week'], how='left'
).drop(columns=['posteam'], errors='ignore')

weekly_stats_df = weekly_stats_df.merge(
    def_sack_rate, left_on=['opponent', 'season', 'week'], right_on=['defteam', 'season', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df['team_sack_rate_allowed'] = weekly_stats_df['team_sack_rate_allowed'].fillna(0)
weekly_stats_df['opp_def_sack_rate'] = weekly_stats_df['opp_def_sack_rate'].fillna(0)

qb_count = qb_mask.sum()
print(f"✓ Added QB-specific features for {qb_count} QB-week records:")
print("  - qb_pass_attempts_3wk_avg & 5wk_avg (shifted)")
print("  - qb_rushing_yards_3wk_avg & 5wk_avg (shifted, captures rushing floor)")
print("  - opp_def_sack_rate (shifted season-to-date, matchup pressure)")
print("  - team_sack_rate_allowed (shifted season-to-date, O-line strength proxy)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 4: RB-Specific Features

In [0]:
# ============================================================================
# CATEGORY 4: Running Back (RB) Specific Features
# ============================================================================
print("\nStep 5: Engineering RB-Specific Features...\n")

# --- 4A: Opportunity Share ---
print("  Calculating Opportunity Share...")

# Calculate team totals per week
team_opportunities = pbp.groupby(['posteam', 'season', 'week']).agg({
    'rush_attempt': 'sum',
    'pass_attempt': 'sum'  # Targets come from pass attempts
}).reset_index()
team_opportunities['team_total_opportunities'] = (
    team_opportunities['rush_attempt'] + team_opportunities['pass_attempt']
)
team_opportunities = team_opportunities[['posteam', 'season', 'week', 'team_total_opportunities']]
team_opportunities.columns = ['team', 'season', 'week', 'team_total_opportunities']

# Calculate player opportunities (rush attempts + targets)
weekly_stats_df['player_opportunities'] = (
    weekly_stats_df['rush_attempts'] + weekly_stats_df['targets']
)

# Merge team totals
weekly_stats_df = weekly_stats_df.merge(
    team_opportunities,
    left_on=['recent_team', 'season', 'week'],
    right_on=['team', 'season', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate opportunity share
weekly_stats_df['opportunity_share'] = (
    weekly_stats_df['player_opportunities'] / weekly_stats_df['team_total_opportunities']
).fillna(0)

# SHIFTED 3-week and 5-week rolling averages for RBs only (reset per season)
rb_mask = weekly_stats_df['position'] == 'RB'
weekly_stats_df.loc[rb_mask, 'rb_opportunity_share_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['opportunity_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_opportunity_share_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['opportunity_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 4B: High-Value Touches (HVTs) ---
print("  Calculating High-Value Touches (yardline <= 10)...")

# Count carries and targets inside the 10-yard line
hvt_carries = pbp[
    (pbp['play_type'] == 'run') & 
    (pbp['yardline_100'] <= 10) &
    (pbp['rusher_player_id'].notna())
].groupby(['rusher_player_id', 'season', 'week']).size().reset_index(name='hvt_carries')
hvt_carries.columns = ['player_id', 'season', 'week', 'hvt_carries']

hvt_targets = pbp[
    (pbp['play_type'] == 'pass') & 
    (pbp['yardline_100'] <= 10) &
    (pbp['receiver_player_id'].notna())
].groupby(['receiver_player_id', 'season', 'week']).size().reset_index(name='hvt_targets')
hvt_targets.columns = ['player_id', 'season', 'week', 'hvt_targets']

# Merge HVTs into main dataframe
weekly_stats_df = weekly_stats_df.merge(hvt_carries, on=['player_id', 'season', 'week'], how='left')
weekly_stats_df = weekly_stats_df.merge(hvt_targets, on=['player_id', 'season', 'week'], how='left')
weekly_stats_df['hvt_carries'] = weekly_stats_df['hvt_carries'].fillna(0)
weekly_stats_df['hvt_targets'] = weekly_stats_df['hvt_targets'].fillna(0)
weekly_stats_df['total_hvts'] = weekly_stats_df['hvt_carries'] + weekly_stats_df['hvt_targets']

# SHIFTED 3-week and 5-week rolling averages for RBs only (reset per season)
weekly_stats_df.loc[rb_mask, 'rb_hvts_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['total_hvts']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_hvts_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['total_hvts']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 4C: Snap Share ---
print("  Calculating Snap Share...")

# Snap counts are keyed by PFR player ids; map them to GSIS ids via rosters
pfr_to_gsis = rosters[['player_id', 'pfr_id']].dropna().drop_duplicates('pfr_id')
snaps = snap_counts.merge(pfr_to_gsis, left_on='pfr_player_id', right_on='pfr_id', how='inner')
snaps = snaps.groupby(['player_id', 'season', 'week'])['offense_pct'].max().reset_index()
snaps.columns = ['player_id', 'season', 'week', 'snap_share']

weekly_stats_df = weekly_stats_df.merge(snaps, on=['player_id', 'season', 'week'], how='left')
weekly_stats_df['snap_share'] = weekly_stats_df['snap_share'].fillna(0)

# SHIFTED 3-week and 5-week rolling averages for RBs only (reset per season)
weekly_stats_df.loc[rb_mask, 'rb_snap_share_3wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['snap_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[rb_mask, 'rb_snap_share_5wk_avg'] = (
    weekly_stats_df[rb_mask].groupby(['player_id', 'season'])['snap_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

rb_count = rb_mask.sum()
print(f"\n✓ Added RB-specific features for {rb_count} RB-week records:")
print("  - rb_opportunity_share_3wk_avg & 5wk_avg (shifted, rush attempts + targets / team total)")
print("  - rb_hvts_3wk_avg & 5wk_avg (shifted, high-value touches inside 10-yard line)")
print("  - rb_snap_share_3wk_avg & 5wk_avg (shifted, % of offensive snaps played)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 5: WR/TE-Specific Features

In [0]:
# ============================================================================
# CATEGORY 5: Wide Receiver (WR) / Tight End (TE) Specific Features
# ============================================================================
print("\nStep 6: Engineering WR/TE-Specific Features...\n")

# --- 5A: Target Share ---
print("  Calculating Target Share...")

# Team passing attempts per week (already calculated above in team_opportunities)
team_pass_attempts = pbp.groupby(['posteam', 'season', 'week'])['pass_attempt'].sum().reset_index()
team_pass_attempts.columns = ['team', 'season', 'week', 'team_pass_attempts']

# Merge team passing attempts
weekly_stats_df = weekly_stats_df.merge(
    team_pass_attempts,
    left_on=['recent_team', 'season', 'week'],
    right_on=['team', 'season', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate target share
weekly_stats_df['target_share'] = (
    weekly_stats_df['targets'] / weekly_stats_df['team_pass_attempts']
).fillna(0)

# SHIFTED rolling averages for WR/TE only (reset per season)
wr_te_mask = weekly_stats_df['position'].isin(['WR', 'TE'])

weekly_stats_df.loc[wr_te_mask, 'wr_te_target_share_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['target_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

weekly_stats_df.loc[wr_te_mask, 'wr_te_target_share_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['target_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5B: Air Yards Share ---
print("  Calculating Air Yards Share...")

# Sum player air yards per week
player_air_yards = pbp[
    pbp['receiver_player_id'].notna() & 
    pbp['air_yards'].notna()
].groupby(['receiver_player_id', 'season', 'week'])['air_yards'].sum().reset_index()
player_air_yards.columns = ['player_id', 'season', 'week', 'player_air_yards']

# Sum team air yards per week
team_air_yards = pbp[
    pbp['posteam'].notna() & 
    pbp['air_yards'].notna()
].groupby(['posteam', 'season', 'week'])['air_yards'].sum().reset_index()
team_air_yards.columns = ['team', 'season', 'week', 'team_air_yards']

# Merge player air yards
weekly_stats_df = weekly_stats_df.merge(player_air_yards, on=['player_id', 'season', 'week'], how='left')
weekly_stats_df['player_air_yards'] = weekly_stats_df['player_air_yards'].fillna(0)

# Merge team air yards
weekly_stats_df = weekly_stats_df.merge(
    team_air_yards,
    left_on=['recent_team', 'season', 'week'],
    right_on=['team', 'season', 'week'],
    how='left'
)
weekly_stats_df.drop(columns=['team'], inplace=True, errors='ignore')

# Calculate air yards share
weekly_stats_df['air_yards_share'] = (
    weekly_stats_df['player_air_yards'] / weekly_stats_df['team_air_yards']
).fillna(0)

# SHIFTED 3-week and 5-week rolling averages for WR/TE only (reset per season)
weekly_stats_df.loc[wr_te_mask, 'wr_te_air_yards_share_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['air_yards_share']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[wr_te_mask, 'wr_te_air_yards_share_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['air_yards_share']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5C: WOPR (Weighted Opportunity Rating) ---
print("  Calculating WOPR...")

# Standard WOPR weights: 1.5 * target share + 0.7 * air yards share
weekly_stats_df['wopr'] = (
    1.5 * weekly_stats_df['target_share'] + 0.7 * weekly_stats_df['air_yards_share']
)

weekly_stats_df.loc[wr_te_mask, 'wr_te_wopr_3wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['wopr']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
weekly_stats_df.loc[wr_te_mask, 'wr_te_wopr_5wk_avg'] = (
    weekly_stats_df[wr_te_mask].groupby(['player_id', 'season'])['wopr']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- 5D: Starting QB Historical Efficiency (Adjusted Yards per Attempt) ---
print("  Calculating Starting QB AY/A...")

# AY/A = (pass yards + 20*pass TDs - 45*INTs) / attempts, season-to-date
# through the PREVIOUS week (shifted to prevent leakage). Joined via the
# scheduled starting QB from the schedules data.
qb_hist = weekly_stats_df[weekly_stats_df['pass_attempts'] > 0][
    ['player_id', 'season', 'week', 'passing_yards', 'passing_tds', 'interceptions', 'pass_attempts']
].copy().sort_values(['player_id', 'season', 'week'])

g = qb_hist.groupby(['player_id', 'season'])
qb_hist['cum_yards'] = g['passing_yards'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_tds'] = g['passing_tds'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_ints'] = g['interceptions'].transform(lambda x: x.shift(1).expanding().sum())
qb_hist['cum_atts'] = g['pass_attempts'].transform(lambda x: x.shift(1).expanding().sum())

qb_hist['starting_qb_aya'] = (
    (qb_hist['cum_yards'] + 20 * qb_hist['cum_tds'] - 45 * qb_hist['cum_ints'])
    / qb_hist['cum_atts']
).replace([np.inf, -np.inf], 0).fillna(0)

qb_aya = qb_hist[['player_id', 'season', 'week', 'starting_qb_aya']].rename(columns={'player_id': 'starting_qb_id'})

weekly_stats_df = weekly_stats_df.merge(qb_aya, on=['starting_qb_id', 'season', 'week'], how='left')
weekly_stats_df['starting_qb_aya'] = weekly_stats_df['starting_qb_aya'].fillna(0)

wr_te_count = wr_te_mask.sum()
print(f"\n✓ Added WR/TE-specific features for {wr_te_count} WR/TE-week records:")
print("  - wr_te_target_share_3wk_avg & 5wk_avg (shifted, targets / team pass attempts)")
print("  - wr_te_air_yards_share_3wk_avg & 5wk_avg (shifted, air yards / team air yards)")
print("  - wr_te_wopr_3wk_avg & 5wk_avg (shifted, 1.5*target share + 0.7*air yards share)")
print("  - starting_qb_aya (shifted season-to-date AY/A of the team's starting QB)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 6: Matchup Features (Opponent Defense)

In [0]:
# ============================================================================
# CATEGORY 6: Matchup Features (Opponent Defense)
# ============================================================================
print("\nStep 7: Engineering Matchup Features (Opponent Defense Metrics)...\n")

# Get defensive team from play-by-play
# For each play, the defense is the team that does NOT have the ball (defteam)

# Calculate fantasy points allowed by defense per position per week
print("  Calculating fantasy points allowed by each defense...")

# Get the defense each player faced each week
# Defense = opponent team for that week (we already have 'opponent' from game_context)

# Aggregate fantasy points allowed by defense (opponent) to each position
def_points_allowed = weekly_stats_df.groupby(['opponent', 'season', 'week', 'position'])['fantasy_points_ppr'].sum().reset_index()
def_points_allowed.columns = ['defense', 'season', 'week', 'position', 'fp_allowed_this_week']

# Calculate season-to-date average fantasy points allowed by defense to each position
# This must be SHIFTED to prevent data leakage (and reset each season)
def_points_allowed = def_points_allowed.sort_values(['defense', 'position', 'season', 'week'])

def_points_allowed['def_fp_allowed_cumsum'] = (
    def_points_allowed.groupby(['defense', 'season', 'position'])['fp_allowed_this_week']
    .transform(lambda x: x.shift(1).expanding().sum())
)

def_points_allowed['def_weeks_played'] = (
    def_points_allowed.groupby(['defense', 'season', 'position']).cumcount()
)

# Average = cumulative sum / weeks played (using shifted data)
def_points_allowed['opp_def_ppg_allowed'] = (
    def_points_allowed['def_fp_allowed_cumsum'] / def_points_allowed['def_weeks_played']
).fillna(0)

# Handle division by zero (first week has no history)
def_points_allowed['opp_def_ppg_allowed'] = def_points_allowed['opp_def_ppg_allowed'].replace([np.inf, -np.inf], 0)

# Merge defensive metrics into main dataframe
weekly_stats_df = weekly_stats_df.merge(
    def_points_allowed[['defense', 'season', 'week', 'position', 'opp_def_ppg_allowed']],
    left_on=['opponent', 'season', 'week', 'position'],
    right_on=['defense', 'season', 'week', 'position'],
    how='left'
)
weekly_stats_df.drop(columns=['defense'], inplace=True, errors='ignore')
weekly_stats_df['opp_def_ppg_allowed'] = weekly_stats_df['opp_def_ppg_allowed'].fillna(0)

# --- Opponent defensive efficiency (DVOA substitutes) ---
# DVOA is premium data; per CONTEXT rules we use yards-per-carry allowed and
# yards-per-play allowed instead. Both are SHIFTED season-to-date rates.
print("  Calculating opponent defensive efficiency (YPC / yards-per-play allowed)...")

def shifted_yards_rate(plays, rate_name):
    g = plays.groupby(['defteam', 'season', 'week']).agg(
        yards=('yards_gained', 'sum'),
        n_plays=('yards_gained', 'size')
    ).reset_index().sort_values(['season', 'week'])
    cum_yards = g.groupby(['defteam', 'season'])['yards'].transform(lambda x: x.shift(1).expanding().sum())
    cum_plays = g.groupby(['defteam', 'season'])['n_plays'].transform(lambda x: x.shift(1).expanding().sum())
    g[rate_name] = (cum_yards / cum_plays).fillna(0)
    return g[['defteam', 'season', 'week', rate_name]]

run_plays_def = pbp[pbp['play_type'] == 'run']
all_plays_def = pbp[pbp['play_type'].isin(['run', 'pass'])]

opp_ypc = shifted_yards_rate(run_plays_def, 'opp_def_ypc_allowed')
opp_ypp = shifted_yards_rate(all_plays_def, 'opp_def_ypp_allowed')

weekly_stats_df = weekly_stats_df.merge(
    opp_ypc, left_on=['opponent', 'season', 'week'], right_on=['defteam', 'season', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df = weekly_stats_df.merge(
    opp_ypp, left_on=['opponent', 'season', 'week'], right_on=['defteam', 'season', 'week'], how='left'
).drop(columns=['defteam'], errors='ignore')

weekly_stats_df['opp_def_ypc_allowed'] = weekly_stats_df['opp_def_ypc_allowed'].fillna(0)
weekly_stats_df['opp_def_ypp_allowed'] = weekly_stats_df['opp_def_ypp_allowed'].fillna(0)

print("✓ Added matchup features:")
print("  - opp_def_ppg_allowed (shifted season-to-date avg FP allowed by opponent defense to this position)")
print("  - opp_def_ypc_allowed (shifted season-to-date yards per carry allowed, rush DVOA substitute)")
print("  - opp_def_ypp_allowed (shifted season-to-date yards per play allowed, DVOA substitute)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 7: Supporting Cast Health (WR1/WR2)

In [0]:
# ============================================================================
# CATEGORY 7: Supporting Cast Health (WR1/WR2 availability)
# ============================================================================
print("\nStep 8: Engineering Supporting Cast Health Features...\n")

# Identify each team's WR1/WR2 as the two WRs with the most cumulative targets
# through the PREVIOUS week (leakage-free), then check the injury report.
# Note: in week 1 there is no target history, so the ranking is arbitrary and
# the flag defaults to healthy unless those players are listed Out/Doubtful.
# Build a complete (player, season, week) grid so injured WR1s (who have no
# stat row in weeks they miss) still get ranked from their prior production.
wr_rows = weekly_stats_df[weekly_stats_df['position'] == 'WR'][
    ['player_id', 'season', 'week', 'targets']
]
season_weeks = weekly_stats_df[['season', 'week']].drop_duplicates()
players_seasons = wr_rows[['player_id', 'season']].drop_duplicates()

grid = players_seasons.merge(season_weeks, on='season')
grid = grid.merge(wr_rows, on=['player_id', 'season', 'week'], how='left')
grid['targets'] = grid['targets'].fillna(0)

# Cumulative targets through the PREVIOUS week, per player-season
grid = grid.sort_values('week')
grid['prior_targets'] = (
    grid.groupby(['player_id', 'season'])['targets'].cumsum() - grid['targets']
)

# Rank WRs within each team-week by prior targets (team is per-season)
grid = grid.merge(player_team, on=['player_id', 'season'], how='left')
grid['wr_rank'] = (
    grid.groupby(['recent_team', 'season', 'week'])['prior_targets']
    .rank(method='first', ascending=False)
)
top2_wrs = grid[grid['wr_rank'] <= 2].copy()

# Players ruled Out or Doubtful on that week's injury report
inj_out = injuries[injuries['report_status'].isin(['Out', 'Doubtful'])][
    ['gsis_id', 'season', 'week']
].drop_duplicates()
inj_out['is_out'] = 1

top2_wrs = top2_wrs.merge(
    inj_out, left_on=['player_id', 'season', 'week'], right_on=['gsis_id', 'season', 'week'], how='left'
)
top2_wrs['is_out'] = top2_wrs['is_out'].fillna(0)

team_wr_health = top2_wrs.groupby(['recent_team', 'season', 'week'])['is_out'].sum().reset_index()
team_wr_health['wr1_wr2_healthy'] = (team_wr_health['is_out'] == 0).astype(int)

weekly_stats_df = weekly_stats_df.merge(
    team_wr_health[['recent_team', 'season', 'week', 'wr1_wr2_healthy']],
    on=['recent_team', 'season', 'week'],
    how='left'
)
weekly_stats_df['wr1_wr2_healthy'] = weekly_stats_df['wr1_wr2_healthy'].fillna(1).astype(int)

print("✓ Added supporting cast health features:")
print("  - wr1_wr2_healthy (1 = neither of the team's top-2 WRs is Out/Doubtful this week)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## CATEGORY 8: Depth Chart Rank (and QB/TE starter filter)

In [0]:
# ============================================================================
# CATEGORY 8: Depth Chart Rank (and QB/TE starter filter)
# ============================================================================
print("\nStep 9: Engineering Depth Chart Features...\n")

# Depth chart data comes in two schemas:
#   2023-2024: weekly published lists (formation/position/depth_team rank)
#   2025+:     daily snapshots (dt/pos_abb/pos_rank)
# Both are pre-game information, so both are leakage-free.
skill_positions = ['QB', 'RB', 'WR', 'TE', 'FB']

# --- Old weekly schema (2023-2024): direct join on (player, season, week) ---
d_old = depth_charts_weekly[
    (depth_charts_weekly['formation'] == 'Offense') &
    (depth_charts_weekly['position'].isin(skill_positions))
].copy()
d_old['depth_team'] = pd.to_numeric(d_old['depth_team'], errors='coerce')
d_old = (
    d_old.groupby(['gsis_id', 'season', 'week'])['depth_team'].min().reset_index()
    .rename(columns={'gsis_id': 'player_id', 'depth_team': 'rank_weekly'})
)
d_old['player_id'] = d_old['player_id'].astype(str)
weekly_stats_df['player_id'] = weekly_stats_df['player_id'].astype(str)

weekly_stats_df = weekly_stats_df.merge(
    d_old, on=['player_id', 'season', 'week'], how='left'
)

# --- New snapshot schema (2025+): as-of join on gameday ---
depth_off = depth_charts_snapshots[depth_charts_snapshots['pos_abb'].isin(skill_positions)].copy()
depth_off['snapshot_dt'] = pd.to_datetime(depth_off['dt']).dt.tz_localize(None).dt.normalize()
depth_off = (
    depth_off.groupby(['gsis_id', 'snapshot_dt'])['pos_rank'].min().reset_index()
    .rename(columns={'gsis_id': 'player_id', 'pos_rank': 'rank_snapshot'})
    .sort_values('snapshot_dt')
)
depth_off['player_id'] = depth_off['player_id'].astype(str)

has_game = weekly_stats_df['gameday'].notna()
matched = pd.merge_asof(
    weekly_stats_df[has_game].sort_values('gameday'),
    depth_off,
    left_on='gameday',
    right_on='snapshot_dt',
    by='player_id',
    direction='backward'
)
matched.drop(columns=['snapshot_dt'], inplace=True, errors='ignore')
weekly_stats_df = pd.concat([matched, weekly_stats_df[~has_game]], ignore_index=True)

# --- Combine the two eras ---
weekly_stats_df['depth_chart_rank'] = weekly_stats_df['rank_weekly'].fillna(
    weekly_stats_df.get('rank_snapshot')
)
weekly_stats_df.drop(columns=['rank_weekly', 'rank_snapshot'], inplace=True, errors='ignore')

# Carry the last known rank forward over gaps (byes, missing playoff lists),
# then give players never on an offensive depth chart a deep sentinel rank
weekly_stats_df = weekly_stats_df.sort_values(['player_id', 'season', 'week'])
weekly_stats_df['depth_chart_rank'] = (
    weekly_stats_df.groupby(['player_id', 'season'])['depth_chart_rank'].ffill()
)
weekly_stats_df['depth_chart_rank'] = weekly_stats_df['depth_chart_rank'].fillna(9).astype(int)

print("✓ Added depth_chart_rank (1 = starter, from latest pre-game depth chart snapshot)")
print(weekly_stats_df.groupby('position')['depth_chart_rank'].value_counts().head(12))

# --- Starter filter for QB and TE ---
# QB and TE are effectively one-man positions for fantasy purposes: only the
# depth-chart starter is worth evaluating. RB and WR keep all ranks
# (committees and WR2/WR3 are fantasy-relevant).
before_filter = len(weekly_stats_df)
qb_te_mask_filter = weekly_stats_df['position'].isin(['QB', 'TE'])
weekly_stats_df = weekly_stats_df[
    ~qb_te_mask_filter | (weekly_stats_df['depth_chart_rank'] == 1)
].reset_index(drop=True)

print(f"\n✓ Filtered to depth-1 starters at QB/TE:")
print(f"  {before_filter:,} → {len(weekly_stats_df):,} records ({before_filter - len(weekly_stats_df):,} backup QB/TE weeks removed)")
print(f"\nCurrent shape: {weekly_stats_df.shape}")

## Final Cleanup and Output Gold DataFrame

In [0]:
# ============================================================================
# FINAL CLEANUP: Fill NaNs and Prepare Gold DataFrame
# ============================================================================
print("\nStep 10: Final Cleanup...\n")

# List of all feature columns that might have NaNs
feature_cols = [
    'fantasy_points_3wk_avg', 'fantasy_points_5wk_avg',
    'qb_pass_attempts_3wk_avg', 'qb_pass_attempts_5wk_avg',
    'qb_rushing_yards_3wk_avg', 'qb_rushing_yards_5wk_avg',
    'rb_opportunity_share_3wk_avg', 'rb_opportunity_share_5wk_avg',
    'rb_hvts_3wk_avg', 'rb_hvts_5wk_avg',
    'rb_snap_share_3wk_avg', 'rb_snap_share_5wk_avg',
    'wr_te_target_share_3wk_avg', 'wr_te_target_share_5wk_avg',
    'wr_te_air_yards_share_3wk_avg', 'wr_te_air_yards_share_5wk_avg',
    'wr_te_wopr_3wk_avg', 'wr_te_wopr_5wk_avg',
    'starting_qb_aya',
    'opp_def_ppg_allowed', 'opp_def_ypc_allowed', 'opp_def_ypp_allowed',
    'opp_def_sack_rate', 'team_sack_rate_allowed',
    'implied_total', 'team_spread', 'team_win_prob', 'temp', 'wind', 'rest_advantage',
    'is_dome', 'snap_share', 'depth_chart_rank',
    'prev_season_ppg', 'prev_season_games'
]

# Fill NaN values with 0
for col in feature_cols:
    if col in weekly_stats_df.columns:
        weekly_stats_df[col] = weekly_stats_df[col].fillna(0)

# Create the Gold DataFrame
gold_df = weekly_stats_df.copy()

print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)
print(f"\nGold DataFrame Shape: {gold_df.shape}")
print(f"Total Features: {len(gold_df.columns)}")
print(f"\nSample of engineered features:")

# Show a sample of key features
feature_sample_cols = [
    'week', 'player_name', 'position', 'recent_team', 'opponent',
    'fantasy_points_ppr', 'fantasy_points_3wk_avg',
    'implied_total', 'is_home', 'opp_def_ppg_allowed'
]
available_cols = [col for col in feature_sample_cols if col in gold_df.columns]
display(gold_df[available_cols].head(10))

print(f"\n✓ All NaN values in feature columns filled with 0")
print(f"✓ Gold DataFrame ready for XGBoost training!")
print(f"\nNext steps:")
print("  1. Split data into train/validation/test sets (by week or chronologically)")
print("  2. Define target variable (fantasy_points_ppr)")
print("  3. Train XGBoost model")
print("  4. Evaluate predictions")

## Feature Summary and Data Quality Checks

In [0]:
# ============================================================================
# DATA QUALITY CHECKS AND FEATURE SUMMARY
# ============================================================================
print("\n" + "="*80)
print("FEATURE SUMMARY & DATA QUALITY REPORT")
print("="*80)

# 1. Overall Statistics
print(f"\n1. OVERALL STATISTICS:")
print(f"   Total Records: {len(gold_df):,}")
print(f"   Total Features: {len(gold_df.columns)}")
print(f"   Seasons Covered: {sorted(gold_df['season'].unique())}")
print(f"   Weeks Covered: {gold_df['week'].min()} - {gold_df['week'].max()}")
print(f"   Unique Players: {gold_df['player_id'].nunique():,}")

# 2. Position Breakdown
print(f"\n2. POSITION BREAKDOWN:")
print(gold_df['position'].value_counts())

# 3. Feature Categories
print(f"\n3. FEATURE CATEGORIES:")
print(f"   ✓ General Context: implied_total, team_spread, team_win_prob, is_home, temp, wind, is_bad_weather, is_dome, rest_advantage")
print(f"   ✓ Rolling Averages: fantasy_points_3wk_avg, fantasy_points_5wk_avg")
print(f"   ✓ Season Carryover: prev_season_ppg, prev_season_games (cold-start prior for week 1)")
print(f"   ✓ QB Features: qb_pass_attempts_3wk/5wk_avg, qb_rushing_yards_3wk/5wk_avg, opp_def_sack_rate, team_sack_rate_allowed")
print(f"   ✓ RB Features: rb_opportunity_share_3wk/5wk_avg, rb_hvts_3wk/5wk_avg, rb_snap_share_3wk/5wk_avg")
print(f"   ✓ WR/TE Features: wr_te_target_share_3wk/5wk_avg, wr_te_air_yards_share_3wk/5wk_avg, wr_te_wopr_3wk/5wk_avg, starting_qb_aya")
print(f"   ✓ Matchup Features: opp_def_ppg_allowed, opp_def_ypc_allowed, opp_def_ypp_allowed")
print(f"   ✓ Supporting Cast: wr1_wr2_healthy")
print(f"   ✓ Depth Chart: depth_chart_rank (QB/TE rows filtered to depth-1 starters only)")

# 4. Missing Values Check
print(f"\n4. MISSING VALUES CHECK:")
missing = gold_df.isnull().sum()
if missing.sum() == 0:
    print("   ✓ No missing values in any column!")
else:
    print("   Columns with missing values:")
    print(missing[missing > 0])

# 5. Data Leakage Prevention Verification
print(f"\n5. DATA LEAKAGE PREVENTION VERIFICATION:")
print("   All rolling averages and historical metrics use .shift(1)")
print("   Week N predictions can only see data from Weeks 1 through N-1")
print("   ✓ Model is ready for time-series cross-validation")

# 6. Sample Feature Values for Top Performers
print(f"\n6. SAMPLE: Top 5 Fantasy Performances with Features")
top_performers = gold_df.nlargest(5, 'fantasy_points_ppr')[[
    'week', 'player_name', 'position', 'fantasy_points_ppr', 
    'fantasy_points_3wk_avg', 'implied_total', 'is_home', 'opp_def_ppg_allowed'
]]
display(top_performers)

print("\n" + "="*80)
print("Gold DataFrame is ready for ML model training!")
print("="*80)

## Persist Gold Table to Unity Catalog

Write `gold_df` to Unity Catalog Delta table `fantasy_football.gold.player_weeks` so downstream model notebooks can read it via Spark.

In [0]:
# Convert pandas DataFrame to Spark DataFrame and write to Delta table
spark_df = spark.createDataFrame(gold_df)

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fantasy_football.gold.player_weeks")

print(f"✓ Saved gold_df ({gold_df.shape[0]:,} rows x {gold_df.shape[1]} cols) to fantasy_football.gold.player_weeks")
print(f"  Table location: {spark.sql('DESCRIBE DETAIL fantasy_football.gold.player_weeks').select('location').first()[0]}")